In [ ]:
%python
# ============================================================
# CONFIGURATION — update these two values before running
# ============================================================
catalog = "<your_catalog_name>"  # e.g. "main"
schema  = "<your_schema_name>"   # e.g. "abac_demo"

print(f"✓ Using: {catalog}.{schema}")


# Tutorial 3: Dynamic Access Control with Mapping Tables

This tutorial demonstrates how to use a mapping table to control both **which rows** a user sees and **how sensitive columns appear** — without creating a group for every combination.

**Pattern:** A mapping table stores each user's attributes (region, department, PII clearance). UDFs call `current_user()`, look up the user's attributes, and apply row filtering and column masking accordingly.

**Requirements:**

- Databricks Runtime 16.4+ or serverless compute
- A Unity Catalog-enabled workspace
- Governed tags created via the Catalog Explorer UI (see Setup section)
- `CREATE SCHEMA`, `CREATE TABLE`, `CREATE FUNCTION` privileges on the target catalog
- Ownership or `MANAGE` privilege on the schema to create policies
- `APPLY TAG` privilege on columns
- `ASSIGN` privilege on the governed tags being used
- `MODIFY` privilege on the mapping table (for the dynamic update step)
- Account group: `abac_tut_admins`

## Why Mapping Tables?

Suppose you want each user to see only rows matching **their region** and **their department**, with PII columns masked based on their clearance level. Without a mapping table, you'd need a group for every region × department combination:

| | Engineering | Sales | Marketing | Finance |
|---|---|---|---|---|
| **US East** | `us_east_eng` | `us_east_sales` | `us_east_mktg` | `us_east_fin` |
| **US West** | `us_west_eng` | `us_west_sales` | `us_west_mktg` | `us_west_fin` |
| **EU** | `eu_eng` | `eu_sales` | `eu_mktg` | `eu_fin` |
| **APAC** | `apac_eng` | `apac_sales` | `apac_mktg` | `apac_fin` |

That's 16 groups for just 4 regions × 4 departments — and it gets worse with every new dimension. Each new region or department requires creating groups AND rewriting UDFs. Layer PII access levels on top (full, masked, none) and the group count triples.

**The mapping table approach:** Store each user's region, department, and PII clearance in a single table. One row per user, one column per attribute. Row filter and column mask UDFs both look up `current_user()` from the same table. To change someone's access, update a row — no groups to manage, no policies to rewrite.

## Setup

> **Note:** This tutorial uses `{catalog}` as the catalog name. Replace with your own catalog if needed.

### Create account group

This tutorial only requires one account group: `abac_tut_admins` (used in the EXCEPT clause so admins bypass filters and masks).

Account-level groups must be created in the **Account Console** (Admin > Groups), not via SQL.

> If this group already exists from a previous tutorial, skip this step.

### Create governed tags

Create the following governed tags in the Catalog Explorer UI (**Catalog** > **Governed Tags** > **Create governed tag**):

| Tag Key | Allowed Values |
|---------|---------------|
| `abac_region` | *(none — key-only tag)* |
| `abac_department` | *(none — key-only tag)* |
| `abac_pii` | `ssn, email, name, phone, address` |

> If `abac_region` or `abac_pii` already exist from previous tutorials, you may only need to create `abac_department` and add any missing allowed values to `abac_pii`.

In [ ]:
spark.sql(f"""
-- Grant base access so all users can query tables in this schema.
-- ABAC policies handle the fine-grained row filtering and column masking on top.
GRANT USE CATALOG ON CATALOG {catalog} TO `account users`
""")

spark.sql(f"""
GRANT USE SCHEMA ON SCHEMA {catalog}.{schema} TO `account users`
""")

spark.sql(f"""
GRANT SELECT ON SCHEMA {catalog}.{schema} TO `account users`
""")


## Step 1: Sample Data

An orders table with `region` and `department` columns for row filtering, plus PII columns (`customer_name`, `customer_email`) for column masking. We tag all four columns so ABAC policies can identify them.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.orders (
  order_id INT,
  customer_name STRING,
  customer_email STRING,
  region STRING,
  department STRING,
  amount DOUBLE,
  order_date DATE
)
""")

spark.sql(f"""
INSERT INTO {catalog}.{schema}.orders VALUES
  (1,  'Acme Corp',     'orders@acme.com',    'us_east', 'engineering', 50000,  '2025-01-15'),
  (2,  'Beta Inc',      'sales@beta.com',     'us_east', 'sales',       75000,  '2025-02-01'),
  (3,  'Gamma LLC',     'info@gamma.com',     'us_west', 'engineering', 30000,  '2025-01-20'),
  (4,  'Delta Co',      'deals@delta.com',    'us_west', 'sales',       95000,  '2025-03-01'),
  (5,  'Epsilon GmbH',  'kontakt@epsilon.de', 'eu',      'engineering', 45000,  '2025-02-15'),
  (6,  'Zeta SA',       'contact@zeta.fr',    'eu',      'sales',       62000,  '2025-01-30'),
  (7,  'Eta Ltd',       'hello@eta.sg',       'apac',    'marketing',   28000,  '2025-03-10'),
  (8,  'Theta Corp',    'biz@theta.com',      'us_east', 'marketing',   55000,  '2025-02-20'),
  (9,  'Iota KK',       'info@iota.jp',       'apac',    'engineering', 41000,  '2025-01-25'),
  (10, 'Kappa Inc',     'sales@kappa.com',    'us_west', 'marketing',   33000,  '2025-03-05')
""")

spark.sql(f"""
-- Tag columns for row filtering
ALTER TABLE {catalog}.{schema}.orders ALTER COLUMN region SET TAGS ('abac_tut_region' = '')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.orders ALTER COLUMN department SET TAGS ('abac_tut_department' = '')
""")

spark.sql(f"""
-- Tag columns for column masking
ALTER TABLE {catalog}.{schema}.orders ALTER COLUMN customer_name SET TAGS ('abac_tut_pii' = 'name')
""")

spark.sql(f"""
ALTER TABLE {catalog}.{schema}.orders ALTER COLUMN customer_email SET TAGS ('abac_tut_pii' = 'email')
""")


## Step 2: The Mapping Table

One table, one row per user, one column per attribute. This single table drives both row filtering (via `region` and `department`) and column masking (via `pii_access`).

The `pii_access` column controls how PII columns appear:
- `full` — see the actual value
- `masked` — see a partial value (e.g., `A***`)
- `none` — see `***REDACTED***`

If a user needs access to multiple region/department combinations, add additional rows (shown in Step 7).

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.user_access (
  user_email STRING,
  region STRING,
  department STRING,
  pii_access STRING  -- 'full', 'masked', or 'none'
)
""")

spark.sql(f"""
-- The first row uses current_user() so you can test immediately.
-- Replace the placeholder emails with actual user emails in your workspace.
INSERT INTO {catalog}.{schema}.user_access VALUES
  (current_user(),      'us_east', 'engineering', 'masked'),
  ('bob@example.com',   'us_west', 'sales',       'full'),
  ('carol@example.com', 'eu',      'engineering', 'none'),
  ('david@example.com', 'apac',    'marketing',   'masked')
""")


In [ ]:
spark.sql(f"""
SELECT * FROM {catalog}.{schema}.user_access
""").display()


## Step 3: Row Filter UDF

The UDF receives a row's region and department values (passed by the policy), looks up the current user in the mapping table, and returns `TRUE` only if there's a matching entry.

Users not in the mapping table see no rows (fail-closed).

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.access_filter(
  region_val STRING,
  dept_val STRING
)
RETURNS BOOLEAN
RETURN EXISTS (
  SELECT 1 FROM {catalog}.{schema}.user_access
  WHERE user_email = current_user()
    AND region = region_val
    AND department = dept_val
)
""")


## Step 4: Column Mask UDF

The mask UDF also reads from the mapping table. It takes the column value and the PII type (passed by the policy), checks the current user's `pii_access` level, and applies type-appropriate masking:

- **Names:** first character + `***` (e.g., `Acme Corp` → `A***`)
- **Emails:** first character + `***@` + domain (e.g., `orders@acme.com` → `o***@acme.com`)

If a user has multiple rows (multi-region access), the highest clearance across all rows applies.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {catalog}.{schema}.pii_mask(
  val STRING,
  pii_type STRING
)
RETURNS STRING
RETURN CASE
  WHEN EXISTS (
    SELECT 1 FROM {catalog}.{schema}.user_access
    WHERE user_email = current_user() AND pii_access = 'full'
  ) THEN val
  WHEN EXISTS (
    SELECT 1 FROM {catalog}.{schema}.user_access
    WHERE user_email = current_user() AND pii_access = 'masked'
  ) THEN
    CASE pii_type
      WHEN 'email' THEN CONCAT(LEFT(val, 1), '***@', SUBSTRING_INDEX(val, '@', -1))
      WHEN 'name'  THEN CONCAT(LEFT(val, 1), '***')
      ELSE CONCAT(LEFT(val, 1), '***')
    END
  ELSE '***REDACTED***'
END
""")


## Step 5: Create the Policies

Three policies, all driven by the same mapping table:
- **Row filter** — matches columns tagged `abac_region` and `abac_department`, passes their values to the filter UDF
- **Column mask (name)** — matches columns tagged `abac_pii = 'name'`, passes `'name'` to the mask UDF
- **Column mask (email)** — matches columns tagged `abac_pii = 'email'`, passes `'email'` to the mask UDF

Using `has_tag_value()` (instead of `has_tag()`) lets each policy target a specific PII type and pass it to the UDF so masking is type-appropriate.

In [ ]:
spark.sql(f"""
-- Row filter: only show rows matching the user's region and department
CREATE OR REPLACE POLICY user_access_filter
ON SCHEMA {catalog}.{schema}
ROW FILTER {catalog}.{schema}.access_filter
TO `account users` EXCEPT abac_tut_admins
FOR TABLES
MATCH COLUMNS has_tag('abac_tut_region') AS r, has_tag('abac_tut_department') AS d
USING COLUMNS (r, d)
""")


In [ ]:
spark.sql(f"""
-- Column mask for name columns
CREATE OR REPLACE POLICY pii_mask_name
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.pii_mask
TO `account users` EXCEPT abac_tut_admins
FOR TABLES
MATCH COLUMNS has_tag_value('abac_tut_pii', 'name') AS m
ON COLUMN m
USING COLUMNS ('name')
""")

spark.sql(f"""
-- Column mask for email columns
CREATE OR REPLACE POLICY pii_mask_email
ON SCHEMA {catalog}.{schema}
COLUMN MASK {catalog}.{schema}.pii_mask
TO `account users` EXCEPT abac_tut_admins
FOR TABLES
MATCH COLUMNS has_tag_value('abac_tut_pii', 'email') AS m
ON COLUMN m
USING COLUMNS ('email')
""")


In [ ]:
spark.sql(f"""
-- With your mapping entry (current_user(), 'us_east', 'engineering', 'masked'),
-- you should see only order #1, with customer_name and customer_email partially masked
SELECT * FROM {catalog}.{schema}.orders
""")


### Expected Results

| User | Region | Department | PII Access | Visible Orders | customer_name | customer_email |
|------|--------|------------|------------|----------------|---------------|----------------|
| *(you)* | us_east | engineering | masked | #1 | `A***` | `o***@acme.com` |
| bob@example.com | us_west | sales | full | #4 | `Delta Co` | `deals@delta.com` |
| carol@example.com | eu | engineering | none | #5 | `***REDACTED***` | `***REDACTED***` |
| david@example.com | apac | marketing | masked | #7 | `E***` | `h***@eta.sg` |
| *(admins)* | — | — | — | All 10 | *(unmasked)* | *(unmasked)* |
| *(unlisted user)* | — | — | — | No orders | — | — |

## Step 6: Dynamic Access Updates

The key benefit: change access by updating the mapping table — no policy or UDF changes needed.

### Reassign a user to a different department

In [ ]:
spark.sql(f"""
-- Move yourself from engineering to sales
UPDATE {catalog}.{schema}.user_access
SET department = 'sales'
WHERE user_email = current_user()
""")


In [ ]:
spark.sql(f"""
-- You now see us_east + sales (order #2, Beta Inc) instead of us_east + engineering
-- PII is still masked since pii_access hasn't changed
SELECT * FROM {catalog}.{schema}.orders
""")


In [ ]:
spark.sql(f"""
-- Revert
UPDATE {catalog}.{schema}.user_access
SET department = 'engineering'
WHERE user_email = current_user()
""")


### Upgrade PII clearance

In [ ]:
spark.sql(f"""
-- Upgrade your PII access from 'masked' to 'full'
UPDATE {catalog}.{schema}.user_access
SET pii_access = 'full'
WHERE user_email = current_user()
""")


In [ ]:
spark.sql(f"""
-- Same row (order #1), but now customer_name and customer_email are fully visible
SELECT * FROM {catalog}.{schema}.orders
""")


In [ ]:
spark.sql(f"""
-- Revert
UPDATE {catalog}.{schema}.user_access
SET pii_access = 'masked'
WHERE user_email = current_user()
""")


### Grant access to an additional region

If a user needs to see data from multiple regions, add another row. No new groups or policies required.

In [ ]:
spark.sql(f"""
-- Grant yourself access to EU engineering as well
INSERT INTO {catalog}.{schema}.user_access
VALUES (current_user(), 'eu', 'engineering', 'masked')
""")


In [ ]:
spark.sql(f"""
-- You now see both us_east + engineering (#1) AND eu + engineering (#5)
SELECT * FROM {catalog}.{schema}.orders
""")


In [ ]:
spark.sql(f"""
-- Remove the additional access
DELETE FROM {catalog}.{schema}.user_access
WHERE user_email = current_user() AND region = 'eu'
""")


### Delete governed tags

If you no longer need them, delete `abac_department` from the Catalog Explorer UI (**Governed Tags** section). Note that `abac_region` and `abac_pii` are shared with other tutorials — only delete them if you're done with all tutorials.

---
## Industry Examples

> **Instructions:** Replace `{catalog}` with your Unity Catalog catalog name and `{schema}` with your target schema name before running.
>
> Groups referenced in this section (`abac_tut_admins`) must be created in the **Account Console** by an admin.
>
> Governed tag `gov_clearance_level` (key-only) must be created in the Catalog Explorer UI.

## Groups Used in Industry Examples

The industry examples below reuse the same account groups from the core tutorial:

| Group | Used As |
|-------|---------|
| `abac_tut_us_team` | Regional (US/North/Voice) |
| `abac_tut_eu_team` | Regional (EU/South/Data) |
| `abac_tut_apac_team` | Regional (APAC/East/Bundle) |
| `abac_tut_admins` | Admin exemption (all policies) |
| `abac_tut_hr_team` | HR domain / Identity owners |
| `abac_tut_finance_team` | Finance domain / Billing / Fraud analysts |
| `abac_tut_marketing_team` | Marketing domain / CPNI owners |

> These groups are managed by your account admin. No group creation SQL is needed here.

## Government — Security Clearance Level Access via Mapping Table

Government agencies manage classified documents with strict need-to-know access controls. A clearance hierarchy (UNCLASSIFIED < CONFIDENTIAL < SECRET < TOP_SECRET) means each user can see documents up to and including their own clearance level.

The mapping table approach is ideal here: instead of creating a group per clearance level, store each analyst's clearance and authorized agency in a lookup table. The UDF enforces the hierarchy dynamically.

**Compliance context:** Executive Order 13526 (Classified National Security Information) requires access controls based on formal clearance determinations and need-to-know.

In [ ]:
spark.sql(f"""
GRANT USE CATALOG ON CATALOG {catalog} TO `account users`
""")

spark.sql(f"""
GRANT USE SCHEMA ON SCHEMA {catalog}.{schema} TO `account users`
""")

spark.sql(f"""
GRANT SELECT ON SCHEMA {catalog}.{schema} TO `account users`
""")


In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.classified_documents (
  doc_id              INT,
  title               STRING,
  clearance_level     STRING,   -- TOP_SECRET, SECRET, CONFIDENTIAL, UNCLASSIFIED
  region              STRING,
  agency              STRING,
  content_summary     STRING,
  classification_date DATE
)
""")

spark.sql(f"""
INSERT INTO {catalog}.{schema}.classified_documents VALUES
  (1, 'Project Alpha Overview',       'UNCLASSIFIED',  'domestic', 'DOD',   'General project overview and public milestones',       '2025-01-10'),
  (2, 'Regional Threat Assessment',   'CONFIDENTIAL',  'domestic', 'DHS',   'Summary of regional threat vectors and mitigation',    '2025-02-15'),
  (3, 'Infrastructure Vulnerability', 'CONFIDENTIAL',  'foreign',  'NSA',   'Critical infrastructure risk analysis Q1 2025',        '2025-03-01'),
  (4, 'Signals Intelligence Brief',   'SECRET',        'foreign',  'NSA',   'SIGINT collection summary — restricted distribution',  '2025-03-20'),
  (5, 'Counterterrorism Op Plan',     'SECRET',        'domestic', 'FBI',   'Operational planning document — law enforcement use',  '2025-04-05'),
  (6, 'Strategic Asset Registry',     'TOP_SECRET',    'domestic', 'DOD',   'Complete registry of strategic national assets',       '2025-04-10'),
  (7, 'Alliance Communication Keys',  'TOP_SECRET',    'foreign',  'NSA',   'Cryptographic key material for allied communications', '2025-04-15'),
  (8, 'Public Infrastructure Report', 'UNCLASSIFIED',  'domestic', 'DHS',   'Annual public infrastructure status report',           '2025-01-20')
""")


In [ ]:
spark.sql(f"""
-- Tag clearance_level column for ABAC policy matching
ALTER TABLE {catalog}.{schema}.classified_documents ALTER COLUMN clearance_level SET TAGS ('gov_clearance_level' = '')
""")


### Security Clearance Mapping Table

The mapping table stores each analyst's email, their clearance level, authorized agency, and need-to-know flag. Users not in this table see no documents (fail-closed by design).

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.{schema}.security_clearance_map (
  user_email         STRING,
  clearance_level    STRING,   -- UNCLASSIFIED, CONFIDENTIAL, SECRET, TOP_SECRET
  authorized_agency  STRING,   -- DOD, DHS, NSA, FBI, or ALL
  need_to_know       BOOLEAN
)
""")

spark.sql(f"""
-- Insert current user with CONFIDENTIAL clearance for testing
-- Replace other entries with real analyst emails from your workspace
INSERT INTO {catalog}.{schema}.security_clearance_map VALUES
  (current_user(),         'CONFIDENTIAL', 'ALL',  TRUE),
  ('analyst1@agency.gov',  'SECRET',       'DHS',  TRUE),
  ('analyst2@agency.gov',  'TOP_SECRET',   'NSA',  TRUE),
  ('public@agency.gov',    'UNCLASSIFIED', 'ALL',  FALSE)
""")


In [ ]:
spark.sql(f"""
SELECT * FROM {catalog}.{schema}.security_clearance_map
""").display()


### Clearance Hierarchy Filter UDF

The filter implements the clearance hierarchy: a SECRET-cleared analyst can read SECRET, CONFIDENTIAL, and UNCLASSIFIED documents but NOT TOP_SECRET. The hierarchy is encoded in the CASE statement.

In [ ]:
spark.sql(f"""
-- Clearance hierarchy filter: users see documents at or below their clearance level
-- Users not in the mapping table see nothing (EXISTS returns FALSE)
CREATE OR REPLACE FUNCTION {catalog}.{schema}.clearance_filter(doc_clearance STRING)
RETURNS BOOLEAN
RETURN (
  SELECT CASE
    WHEN ua.clearance_level = 'TOP_SECRET'   THEN TRUE
    WHEN ua.clearance_level = 'SECRET'       AND doc_clearance IN ('SECRET', 'CONFIDENTIAL', 'UNCLASSIFIED') THEN TRUE
    WHEN ua.clearance_level = 'CONFIDENTIAL' AND doc_clearance IN ('CONFIDENTIAL', 'UNCLASSIFIED') THEN TRUE
    WHEN ua.clearance_level = 'UNCLASSIFIED' AND doc_clearance = 'UNCLASSIFIED' THEN TRUE
    ELSE FALSE
  END
  FROM {catalog}.{schema}.security_clearance_map ua
  WHERE ua.user_email = current_user()
    AND ua.need_to_know = TRUE
  LIMIT 1
)
""")


### Schema-Level Clearance Policy

The policy uses `MATCH COLUMNS` on the `gov_clearance_level` tag. Any table added to this schema with a clearance_level column tagged with `gov_clearance_level` automatically inherits this access control.

In [ ]:
spark.sql(f"""
-- Schema-level clearance filter policy
-- abac_tut_admins are exempt so they can audit all documents
CREATE OR REPLACE POLICY clearance_access_policy
ON SCHEMA {catalog}.{schema}
ROW FILTER {catalog}.{schema}.clearance_filter
TO `account users` EXCEPT abac_tut_admins
FOR TABLES
MATCH COLUMNS has_tag('gov_clearance_level') AS clearance_col
USING COLUMNS (clearance_col)
""")


In [ ]:
spark.sql(f"""
-- Verify: with CONFIDENTIAL clearance (current_user() default),
-- you should see docs 1, 2, 3, 8 (UNCLASSIFIED and CONFIDENTIAL only)
-- SECRET and TOP_SECRET documents are hidden
SELECT doc_id, title, clearance_level, agency, content_summary
FROM {catalog}.{schema}.classified_documents
ORDER BY clearance_level, doc_id
""")


**Expected results by clearance level:**

| Clearance | Visible Documents |
|-----------|-------------------|
| UNCLASSIFIED | Docs 1, 8 only |
| CONFIDENTIAL | Docs 1, 2, 3, 8 |
| SECRET | Docs 1, 2, 3, 4, 5, 8 |
| TOP_SECRET | All 8 documents |
| Not in map / need_to_know=FALSE | No documents |

### Dynamic Clearance Upgrade

Demonstrate the mapping table power: upgrade clearance from CONFIDENTIAL to SECRET without touching any policy or UDF.

In [ ]:
spark.sql(f"""
-- Upgrade clearance: CONFIDENTIAL -> SECRET
-- No policy or UDF changes needed
UPDATE {catalog}.{schema}.security_clearance_map
SET clearance_level = 'SECRET'
WHERE user_email = current_user()
""")


In [ ]:
spark.sql(f"""
-- Now you should see 6 documents (adds SECRET docs 4, 5)
SELECT doc_id, title, clearance_level, agency
FROM {catalog}.{schema}.classified_documents
ORDER BY clearance_level, doc_id
""")


In [ ]:
spark.sql(f"""
-- Revert clearance back to CONFIDENTIAL
UPDATE {catalog}.{schema}.security_clearance_map
SET clearance_level = 'CONFIDENTIAL'
WHERE user_email = current_user()
""")
